# PipelineTS Quick Start: Retail Demand Forecasting
# PipelineTS 快速开始：零售销量预测

This tutorial uses a common industrial scenario: **daily store demand forecasting** with promotions, holidays, weather-like signals, and inventory stockout effects.

本教程使用常见工业场景：带促销、节假日、天气类信号和缺货影响的**门店日销量预测**。

You will cover:

你将学习：

- Current model names from `ModelPipeline.list_all_available_models()`
- `ModelPipeline.fit()`, `predict()`, `predict_quantiles()`
- `SmartRouter` fast automation
- Forecast visualization

In [ ]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

def make_retail_demand(n_days=240, n_stores=1, start="2023-01-01"):
    rows = []
    for i in range(n_stores):
        rng = np.random.default_rng(100 + i)
        dates = pd.date_range(start, periods=n_days, freq="D")
        dow = dates.dayofweek.to_numpy()
        month = dates.month.to_numpy()
        holiday = ((dow >= 5) | rng.binomial(1, 0.04, n_days).astype(bool)).astype(int)
        promotion = rng.binomial(1, 0.14 + 0.06 * (dow >= 4), n_days).astype(int)
        price_index = 1.0 + 0.04 * np.sin(np.linspace(0, 5 * np.pi, n_days)) + rng.normal(0, 0.015, n_days)
        temperature = 18 + 10 * np.sin(np.linspace(-0.8, 2.8 * np.pi, n_days)) + rng.normal(0, 1.8, n_days)
        stockout = rng.binomial(1, 0.025, n_days)
        baseline = 120 + 18 * i
        weekly = np.where(dow < 5, 8, 28)
        seasonal = 16 * np.sin(2 * np.pi * np.arange(n_days) / 365.25 + i / 3)
        trend = 0.08 * np.arange(n_days)
        demand = (
            baseline + weekly + seasonal + trend
            + 34 * promotion + 22 * holiday
            + 0.9 * np.maximum(temperature - 20, 0)
            - 75 * (price_index - 1.0)
            - 45 * stockout
            + rng.normal(0, 7, n_days)
        )
        rows.append(pd.DataFrame({
            "date": dates,
            "store_id": f"store_{i + 1:02d}",
            "sales": np.maximum(demand, 1),
            "promotion": promotion,
            "holiday": holiday,
            "price_index": price_index,
            "temperature": temperature,
            "stockout": stockout,
            "month": month,
        }))
    return pd.concat(rows, ignore_index=True)

In [ ]:
data = make_retail_demand(n_days=260, n_stores=1)
data = data.drop(columns=["store_id"])
train = data.iloc[:-14].reset_index(drop=True)
valid = data.iloc[-14:].reset_index(drop=True)
future_covariates = valid[["date", "promotion", "holiday"]].reset_index(drop=True)

print(train.tail(3))
print(valid.head(3))

In [ ]:
from PipelineTS.pipeline import ModelPipeline

print("Available model names:")
print(ModelPipeline.list_all_available_models())

In [ ]:
pipeline = ModelPipeline(
    time_col="date",
    target_col="sales",
    lags=14,
    known_covariates=["promotion", "holiday"],
    past_covariates=["temperature", "price_index", "stockout"],
    include_models=["random_forest", "extra_forest", "multi_output_model"],
    quantile=0.9,
    cv=2,
    random_forest__n_estimators=120,
    extra_forest__n_estimators=120,
)

leaderboard = pipeline.fit(train, valid_data=valid)
leaderboard

In [ ]:
forecast = pipeline.predict(n=14, future_covariates=future_covariates)
forecast.head()

In [ ]:
quantiles = pipeline.predict_quantiles(
    n=14,
    levels=[0.5, 0.8, 0.9, 0.95],
    future_covariates=future_covariates,
)
quantiles.head()

In [ ]:
from PipelineTS.plot import plot_forecast, plot_leaderboard

plot_forecast(train, forecast, time_col="date", target_col="sales", history_tail=90, lang="zh")
plot_leaderboard(leaderboard, lang="zh")

In [ ]:
from PipelineTS.pipeline import SmartRouter

router = SmartRouter(
    time_col="date",
    target_col="sales",
    known_covariates=["promotion", "holiday"],
    past_covariates=["temperature", "price_index", "stockout"],
    preset="fast",
    include_models=["random_forest", "extra_forest", "multi_output_model"],
    quantile=0.9,
    time_limit=45,
)
router.fit(train, valid_data=valid)
router.predict(14, future_covariates=future_covariates).head()